# Bengaluru Weather, Air Quality & Bookstore Scraping

USN: 1RUA25CSE0433
Name: Sharanya V

In [ ]:
USN_NUMBER = "1RUA25CSE0433"
NAME       = "Sharanya V"

print(f"USN: {USN_NUMBER}")
print(f"Name: {NAME}")

## Step 1: Import libraries

In [ ]:
import requests
import pandas as pd
import time
import csv
from bs4 import BeautifulSoup

## Step 2: Define 5 Bengaluru locations

In [ ]:
locations = [
    {"name": "Bengaluru City Center", "lat": 12.9716, "lon": 77.5946},
    {"name": "Whitefield",            "lat": 12.9698, "lon": 77.7500},
    {"name": "Electronic City",       "lat": 12.8452, "lon": 77.6602},
    {"name": "Yeshwanthpur",          "lat": 13.0284, "lon": 77.5540},
    {"name": "Koramangala",           "lat": 12.9352, "lon": 77.6245},
]

for loc in locations:
    print(loc)

## Step 3: Functions to fetch weather and air quality data

In [ ]:
def get_weather(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,relative_humidity_2m"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    hourly = data["hourly"]
    records = []
    for i, timestamp in enumerate(hourly["time"]):
        records.append({
            "time": timestamp,
            "temperature_C": hourly["temperature_2m"][i],
            "humidity_percent": hourly["relative_humidity_2m"][i]
        })
    return records


def get_air_quality(lat, lon):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "pm10,pm2_5,carbon_monoxide"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    hourly = data["hourly"]
    records = []
    for i, timestamp in enumerate(hourly["time"]):
        records.append({
            "time": timestamp,
            "pm10": hourly["pm10"][i],
            "pm2_5": hourly["pm2_5"][i],
            "carbon_monoxide": hourly["carbon_monoxide"][i]
        })
    return records

## Step 4: Loop through all locations and collect data

In [ ]:
weather_records = []
air_quality_records = []

for loc in locations:
    print(f"Fetching data for {loc['name']}...")

    weather_data = get_weather(loc["lat"], loc["lon"])
    for row in weather_data:
        row["location"] = loc["name"]
    weather_records.extend(weather_data)

    aq_data = get_air_quality(loc["lat"], loc["lon"])
    for row in aq_data:
        row["location"] = loc["name"]
    air_quality_records.extend(aq_data)

    time.sleep(1)

print(f"Collected {len(weather_records)} weather rows and {len(air_quality_records)} air quality rows.")

## Step 5: Save weather and air quality data to CSV

In [ ]:
weather_df = pd.DataFrame(weather_records)
air_quality_df = pd.DataFrame(air_quality_records)

weather_df = weather_df[["location", "time", "temperature_C", "humidity_percent"]]
air_quality_df = air_quality_df[["location", "time", "pm10", "pm2_5", "carbon_monoxide"]]

weather_df.to_csv("bengaluru_weather.csv", index=False)
air_quality_df.to_csv("bengaluru_air_quality.csv", index=False)

weather_df.head()

In [ ]:
air_quality_df.head()

## Step 6: Bookstore web scraping

In [ ]:
base_url = "http://books.toscrape.com/catalogue/page-1.html"
books_data = []

current_url = base_url

while current_url:
    print(f"Scraping: {current_url}")
    response = requests.get(current_url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text.strip()
        rating_class = book.find("p", class_="star-rating")["class"]
        rating = [c for c in rating_class if c != "star-rating"][0]

        books_data.append({
            "title": title,
            "price": price,
            "rating": rating
        })

    next_li = soup.find("li", class_="next")
    if next_li:
        next_href = next_li.a["href"]
        current_url = "http://books.toscrape.com/catalogue/" + next_href
        time.sleep(1)
    else:
        current_url = None

print(f"Scraped {len(books_data)} books total.")

## Step 7: Save scraped book data to CSV

In [ ]:
books_df = pd.DataFrame(books_data)
books_df.to_csv("books_data.csv", index=False)

books_df.head(10)